# 03 — m=1 simple-cell dataset

Assembles the 31 manually verified m=1 simple cells from the population CSVs, loads their RF maps from the NWB files, and saves everything to `derived_data/m1_cells/m1_neuron_dataset.pkl`.

In [ ]:
from pathlib import Path
import sys, os

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / 'src'))
sys.path.insert(0, str(REPO_ROOT))

population_dir = REPO_ROOT / 'derived_data' / 'population'
gallery_dir    = REPO_ROOT / 'derived_data' / 'm1_cells'
review_dir     = REPO_ROOT / 'derived_data' / 'review_judgements'
cache_dir      = REPO_ROOT / 'data' / 'cache'
gallery_dir.mkdir(parents=True, exist_ok=True)
import sys, warnings, time, os, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from scipy.ndimage import gaussian_filter
from sklearn.linear_model import RidgeCV, Ridge
from allensdk.core.brain_observatory_cache import BrainObservatoryCache

from rf_analysis.sparse_noise import (
    fit_rf_by_order,
    gaussian_deriv_m1,
    estimate_phi_from_lobes,
    best_display_orientation,
)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

In [ ]:
def batch_get_rf_maps_safe(dataset, requested_ids,
                            n_lags=8, response_delay=4, response_window=5):
    from rf_analysis.sparse_noise import _build_design_matrix, _pixel_size, _LAMBDA_GRID
    exp_cells = list(dataset.get_cell_specimen_ids())
    req_set   = set(int(c) for c in requested_ids)
    valid_ids = [c for c in exp_cells if c in req_set]
    if not valid_ids:
        raise ValueError('No requested cell_ids found in experiment.')

    stim_name = next((s for s in dataset.list_stimuli() if 'sparse_noise' in s), None)
    if stim_name is None:
        raise ValueError('No sparse noise stimulus.')

    stim_table = dataset.get_stimulus_table(stim_name)
    tr         = dataset.get_locally_sparse_noise_stimulus_template(stimulus=stim_name)
    template   = tr[0] if isinstance(tr, tuple) else tr
    n_tf, grid_h, grid_w = template.shape
    on_val, off_val = int(template.max()), int(template.min())
    pix_size        = _pixel_size(stim_name)

    starts        = stim_table['start'].values.astype(int)
    frame_indices = stim_table['frame'].values.astype(int)

    _, all_dff    = dataset.get_dff_traces(cell_specimen_ids=valid_ids)
    n_cells, n_tp = all_dff.shape

    X = _build_design_matrix(template, frame_indices, on_val, off_val,
                              grid_h, grid_w, n_lags)

    win_idx  = (starts + response_delay)[:, None] + np.arange(response_window)
    in_bnds  = (win_idx >= 0) & (win_idx < n_tp)
    win_safe = np.clip(win_idx, 0, n_tp - 1)
    dff_wins = all_dff[:, win_safe]
    dff_wins[:, ~in_bnds] = np.nan
    Y = np.nanmean(dff_wins, axis=2).T
    Y = np.where(np.isnan(Y), 0.0, Y).astype(np.float64)

    frame_valid  = (frame_indices >= 0) & (frame_indices < n_tf)
    X_fit, Y_fit = X[frame_valid].astype(np.float64), Y[frame_valid]
    if X_fit.shape[0] < 20:
        raise ValueError('Too few valid presentations.')

    try:
        rcv = RidgeCV(alphas=_LAMBDA_GRID, cv=None,
                      fit_intercept=True, alpha_per_target=True)
        rcv.fit(X_fit, Y_fit)
        W = np.atleast_2d(rcv.coef_)
    except TypeError:
        rcv = RidgeCV(alphas=_LAMBDA_GRID, cv=None, fit_intercept=True)
        rcv.fit(X_fit, Y_fit.mean(axis=1, keepdims=True))
        r = Ridge(alpha=float(rcv.alpha_), fit_intercept=True)
        r.fit(X_fit, Y_fit)
        W = np.atleast_2d(r.coef_)

    rf_out = {}
    for i, cid in enumerate(valid_ids):
        strf     = W[i].reshape(n_lags, grid_h, grid_w)
        best_lag = int(np.argmax([np.abs(strf[l]).max() for l in range(n_lags)]))
        rf_out[cid] = (strf[best_lag], pix_size)
    return rf_out

In [ ]:
csv_files = sorted(population_dir.glob('rf_params_order_v2_container_*.csv'))

dfs = []
for f in csv_files:
    df = pd.read_csv(f)
    if 'container_id' not in df.columns:
        df['container_id'] = int(f.stem.split('_')[-1])
    dfs.append(df)
all_df = pd.concat(dfs, ignore_index=True)

for col in ['sigma', 'theta', 'kappa', 'r_squared', 'sigma_x', 'sigma_y',
            'x0', 'y0', 'phi', 'phi_confidence', 'theta_hybrid',
            'cortex_x_um', 'cortex_y_um', 'r2_m0', 'r2_m1', 'r2_m2',
            'delta_r2_vs_m0']:
    if col in all_df.columns:
        all_df[col] = pd.to_numeric(all_df[col], errors='coerce')

all_df['sigma_major'] = all_df[['sigma_x', 'sigma_y']].max(axis=1)
all_df['sigma_minor'] = all_df[['sigma_x', 'sigma_y']].min(axis=1)
all_df['log_sigma']   = np.log10(all_df['sigma'])
all_df['log_kappa']   = np.log(all_df['kappa'])

all_df.loc[all_df['derivative_order'] == 0, 'theta_hybrid'] =     all_df.loc[all_df['derivative_order'] == 0, 'theta']
all_df.loc[all_df['derivative_order'] == 2, 'theta_hybrid'] =     all_df.loc[all_df['derivative_order'] == 2, 'theta']

In [ ]:
judged      = {}
nb07_m1_ids = set()
nb07_path   = review_dir / 'manual_m_judgements.csv'
if nb07_path.exists():
    jdf = pd.read_csv(nb07_path)
    jdf = jdf[jdf['m_manual'] >= 0]
    for _, r in jdf.iterrows():
        judged[int(r['cell_id'])] = int(r['m_manual'])
        if int(r['m_manual']) == 1:
            nb07_m1_ids.add(int(r['cell_id']))
else:
    pass

later_sources = [
    (review_dir  / 'nb09_review_judgements.csv',       'nb09',          'm_manual'),
    (population_dir / 'targeted_review_judgements.csv',   'targeted',      'm_manual'),
    (population_dir / 'unreviewed_m1_judgements.csv',     'unreviewed_m1', 'm_manual'),
    (population_dir / 'rescue_review_judgements.csv',     'rescue',        'm_final'),
]
for path, label, col in later_sources:
    if not path.exists():
        continue
    jdf     = pd.read_csv(path)
    jdf     = jdf[jdf[col] >= 0]
    applied = skipped = 0
    for _, r in jdf.iterrows():
        cid   = int(r['cell_id'])
        m_new = int(r[col])
        if cid in nb07_m1_ids and m_new == 0:
            skipped += 1
            continue
        judged[cid] = m_new
        applied += 1

all_df['m_manual']          = all_df['cell_id'].map(judged)
all_df['manually_verified'] = all_df['cell_id'].isin(judged)
all_df['m_final'] = np.where(
    all_df['m_manual'].notna(),
    all_df['m_manual'],
    all_df['derivative_order']
).astype(int)

if 'delta_r2_vs_m0' in all_df.columns:
    demote = (
        (all_df['m_final'] == 2) &
        (~all_df['manually_verified']) &
        (all_df['delta_r2_vs_m0'] < 0.10)
    )
    all_df.loc[demote, 'm_final'] = 0

m1 = all_df[all_df['m_final'] == 1].copy()

In [ ]:
rf_map_cache  = {}
dataset_cache = {}

by_container = m1.groupby('container_id')['cell_id'].apply(list).to_dict()

t_start = time.time()
for i, (container_id, cids) in enumerate(by_container.items()):
    if container_id not in container_to_exp:
        continue
    try:
        if container_id not in dataset_cache:
            dataset_cache[container_id] = boc.get_ophys_experiment_data(
                container_to_exp[container_id])
        maps = batch_get_rf_maps_safe(dataset_cache[container_id], cids)
        rf_map_cache.update(maps)
    except Exception as e:
        pass


In [ ]:
from scipy.ndimage import gaussian_filter as _gf

N_RANDOM_STARTS = 15
SMOOTH_SIGMA    = 0.75

fit_results = {}

for idx, row in m1.reset_index(drop=True).iterrows():
    cid = int(row['cell_id'])
    if cid not in rf_map_cache:
        continue
    rf_map, pix = rf_map_cache[cid]
    rf_on  = np.maximum(rf_map,  0.0)
    rf_off = np.maximum(-rf_map, 0.0)
    grid_h, grid_w = rf_map.shape
    Y_g, X_g = np.mgrid[0:grid_h, 0:grid_w].astype(float)
    xy = (X_g, Y_g)
    try:
        p, q = fit_rf_by_order(
            rf_on, rf_off, m=1,
            pixel_size_deg=pix,
            smooth_sigma=SMOOTH_SIGMA,
            n_random_starts=N_RANDOM_STARTS,
            seed=cid,
        )

        phi_fresh, phi_conf = estimate_phi_from_lobes(
            rf_on, rf_off, m=1,
            pixel_size_deg=pix,
            smooth_sigma=SMOOTH_SIGMA,
        )

        su_px = p['sigma_x'] / pix
        sv_px = p['sigma_y'] / pix
        x0_px = p['x0']     / pix
        y0_px = p['y0']     / pix
        A_abs = abs(p['amplitude'])

        rf_smooth_corr = _gf(rf_map, sigma=SMOOTH_SIGMA)

        disp = best_display_orientation(
            rf_smooth_corr, x0_px, y0_px, su_px, sv_px, m=1,
            extra_angles_deg=(p['theta'], phi_fresh),
        )
        fitted             = disp['model_map'].reshape(grid_h, grid_w)
        best_angle         = disp['theta_display']
        best_corr          = disp['corr']
        angle_corr_profile = disp['angle_corr_profile']

        fit_results[cid] = (p, q, fitted, pix, phi_fresh, phi_conf, angle_corr_profile)
    except Exception as e:
        import traceback
        traceback.print_exc()


In [ ]:
NAMES = [
    'Georgette', 'Steve',     'Cassandra', 'Garibaldi', 'Gertrude',
    'Vasily',    'Hildegard', 'Bobby',     'Valentina', 'Ting',
    'Mathilde',  'Crispin',   'Leontine',  'Reginald',  'Rowena',
    'Bertram',   'Millicent', 'Archibald', 'Marina',    'Desmond',
    'Cordelia',  'Cornelius', 'Ottoline',  'Fletcher',  'Xiomara',
    'Wendell',   'Winifred',  'Lysander',  'Isadora',   'Barnabas',
    'Perpetua',
]

sorted_m1 = (
    m1[m1['cell_id'].isin(fit_results)]
    .sort_values('cell_id', ascending=True)
    .reset_index(drop=True)
)
assert len(sorted_m1) == 31, f"Expected 31 neurons in fit_results, got {len(sorted_m1)}"

name_map = {int(row['cell_id']): NAMES[i]
            for i, row in sorted_m1.iterrows()}

SMOOTH_SIGMA_DISPLAY = 1.5

m1_dataset = []

for i, row in sorted_m1.iterrows():
    cid = int(row['cell_id'])
    p, q, model_map, pix, phi_fresh, phi_conf, angle_corr_profile = fit_results[cid]
    rf_raw = rf_map_cache[cid][0].astype(np.float32)

    rf_smooth  = gaussian_filter(rf_raw, sigma=SMOOTH_SIGMA).astype(np.float32)
    rf_display = gaussian_filter(rf_raw, sigma=SMOOTH_SIGMA_DISPLAY).astype(np.float32)

    vmax = float(np.nanpercentile(np.abs(rf_raw), 98))

    phi_deg = (
        float(phi_fresh)
        if (phi_fresh is not None and not np.isnan(float(phi_fresh)))
        else (float(row['theta_hybrid']) if pd.notna(row.get('theta_hybrid', np.nan))
              else float(p['theta']))
    )
    edge_orientation = (phi_deg + 90.0) % 180.0

    model_abs_max = float(np.abs(model_map).max())
    acp = np.asarray(angle_corr_profile, dtype=float)
    orientation_sharpness = (float(np.nanmax(acp) / np.nanmean(acp))
                             if np.isfinite(acp).any() and np.nanmean(acp) > 0
                             else float('nan'))

    record = {
        'name':             NAMES[i],
        'name_rank':        i + 1,
        'cell_id':          cid,
        'container_id':     int(row['container_id']),
        'cre_line':         str(row.get('cre_line', '')),
        'imaging_depth_um': (int(row['imaging_depth'])
                             if pd.notna(row.get('imaging_depth', np.nan)) else None),
        'cortex_x_um':      (float(row['cortex_x_um'])
                             if pd.notna(row.get('cortex_x_um', np.nan)) else None),
        'cortex_y_um':      (float(row['cortex_y_um'])
                             if pd.notna(row.get('cortex_y_um', np.nan)) else None),

        'rf_raw':           rf_raw,
        'rf_smooth':        rf_smooth,
        'rf_display':       rf_display,
        'model_map':        model_map.astype(np.float32),

        'vmax':             vmax,
        'pixel_size_deg':   pix,
        'grid_shape':       rf_raw.shape,
        'x0_px':            float(p['x0'] / pix),
        'y0_px':            float(p['y0'] / pix),

        'sigma_deg':        float(p['sigma']),
        'kappa':            float(p['kappa']),
        'kappa_dir':        float(p['kappa_dir']),
        'sigma_phi_deg':    float(p['sigma_phi']),
        'sigma_orth_deg':   float(p['sigma_orth']),
        'sigma_x_deg':      float(p['sigma_x']),
        'sigma_y_deg':      float(p['sigma_y']),
        'x0_deg':           float(p['x0']),
        'y0_deg':           float(p['y0']),
        'amplitude':        float(p['amplitude']),
        'theta_envelope':   float(p['theta']),
        'phi_deg':          phi_deg,
        'phi_confidence':   float(phi_conf),
        'edge_orientation': edge_orientation,

        'r_squared':          float(q['r_squared']),
        'rmse':               float(q['rmse']),
        'aic':                float(q.get('aic', np.nan)),
        'converged':          bool(q['converged']),
        'n_starts_succeeded': int(p.get('n_starts_succeeded', 0)),
        'r2_improvement':     float(p.get('r2_improvement', 0.0)),

        'delta_r2_vs_m0':   (float(row['delta_r2_vs_m0'])
                             if pd.notna(row.get('delta_r2_vs_m0', np.nan)) else None),
        'r2_m0':            (float(row['r2_m0'])
                             if pd.notna(row.get('r2_m0', np.nan)) else None),
        'r2_m1':            (float(row['r2_m1'])
                             if pd.notna(row.get('r2_m1', np.nan)) else None),
        'r2_m2':            (float(row['r2_m2'])
                             if pd.notna(row.get('r2_m2', np.nan)) else None),

        'low_kappa':         bool(p['kappa'] < 1.3),
        'low_r2':            bool(q['r_squared'] < 0.45),
        'multistart_helped': bool(p.get('r2_improvement', 0.0) > 0.05),

        'contour_fracs':    [0.3, 0.6, 0.9],
        'model_abs_max':    model_abs_max,
        'angle_corr_profile':    [float(v) for v in acp],
        'orientation_sharpness': orientation_sharpness,

        'comment':          '',
    }
    m1_dataset.append(record)

dataset_pkl = gallery_dir / 'm1_neuron_dataset.pkl'
dataset_csv = gallery_dir / 'm1_neuron_dataset.csv'

with open(dataset_pkl, 'wb') as f:
    pickle.dump(m1_dataset, f, protocol=4)

SCALAR_KEYS = [
    'name', 'name_rank', 'cell_id', 'container_id', 'cre_line',
    'imaging_depth_um', 'cortex_x_um', 'cortex_y_um',
    'sigma_deg', 'kappa', 'kappa_dir', 'sigma_phi_deg', 'sigma_orth_deg',
    'sigma_x_deg', 'sigma_y_deg',
    'x0_deg', 'y0_deg', 'x0_px', 'y0_px', 'amplitude',
    'theta_envelope', 'phi_deg', 'phi_confidence', 'edge_orientation',
    'r_squared', 'rmse', 'aic', 'converged',
    'n_starts_succeeded', 'r2_improvement',
    'delta_r2_vs_m0', 'r2_m0', 'r2_m1', 'r2_m2',
    'low_kappa', 'low_r2', 'multistart_helped',
    'pixel_size_deg', 'vmax', 'model_abs_max', 'orientation_sharpness', 'comment',
]
pd.DataFrame([{k: r[k] for k in SCALAR_KEYS} for r in m1_dataset]) \
  .to_csv(dataset_csv, index=False)

for r in m1_dataset:
    flags = []
    if r['low_kappa']:         flags.append('low-κ')
    if r['low_r2']:            flags.append('low-R²')
    if r['multistart_helped']: flags.append('multi-start+')


In [ ]:
def draw_orientation_bar(ax, phi_deg, cx, cy, half_len, color='white', lw=1.5):
    """Draw edge orientation bar. phi_deg is the differentiation axis (φ);
    the bar is drawn at φ+90° = preferred edge orientation."""
    edge_deg = (phi_deg + 90) % 180
    th = np.deg2rad(edge_deg)
    dx, dy = np.cos(th) * half_len, np.sin(th) * half_len
    ax.plot([cx - dx, cx + dx], [cy - dy, cy + dy],
            color=color, lw=lw, solid_capstyle='round')

plot_df = (
    m1[m1['cell_id'].isin(fit_results)]
    .sort_values('kappa', ascending=True)
    .reset_index(drop=True)
)
n_neurons = len(plot_df)
N_COLS    = 4
n_rows    = int(np.ceil(n_neurons / N_COLS))

fig = plt.figure(figsize=(N_COLS * 4.2, n_rows * 2.1), facecolor='white')
outer = gridspec.GridSpec(n_rows, N_COLS, figure=fig, hspace=0.28, wspace=0.10)

for idx, row in plot_df.iterrows():
    cid = int(row['cell_id'])
    p, q, fitted, pix, phi_fresh, phi_conf, *_ = fit_results[cid]
    rf_map = rf_map_cache[cid][0]
    name   = name_map.get(cid, '')

    inner = gridspec.GridSpecFromSubplotSpec(
        1, 2, subplot_spec=outer[idx // N_COLS, idx % N_COLS], wspace=0.04)

    vmax = np.nanpercentile(np.abs(rf_map), 98)
    imkw = dict(cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                aspect='equal', origin='lower', interpolation='nearest')

    phi_disp = (float(phi_fresh)
                if (phi_fresh is not None and not np.isnan(float(phi_fresh)))
                else (float(row['theta_hybrid']) if pd.notna(row['theta_hybrid'])
                      else float(p['theta'])))
    r2_col    = '#1a7a4a' if q['r_squared'] >= 0.5 else '#888888'
    cre_short = str(row.get('cre_line', '')).replace(
        'Cux2-CreERT2', 'Cux2').replace('Slc17a7-IRES2-Cre', 'Slc17a7')

    ax_r = fig.add_subplot(inner[0, 0])
    ax_r.imshow(rf_map, **imkw)
    ax_r.set_xticks([]); ax_r.set_yticks([])
    for sp in ax_r.spines.values():
        sp.set_edgecolor('#bbbbbb'); sp.set_linewidth(0.5)
    ax_r.set_title(f'#{idx+1} {name}  {cre_short}', fontsize=6.5,
                   color='#444', pad=2, loc='left')
    ax_r.text(0.04, 0.96, 'raw RF', transform=ax_r.transAxes,
              fontsize=5.5, color='white', va='top',
              bbox=dict(facecolor='#333', alpha=0.65, pad=1, edgecolor='none'))

    ax_f = fig.add_subplot(inner[0, 1])
    ax_f.imshow(fitted, **imkw)
    ax_f.set_xticks([]); ax_f.set_yticks([])
    for sp in ax_f.spines.values():
        sp.set_edgecolor('#2196F3'); sp.set_linewidth(1.0)
    draw_orientation_bar(ax_f, phi_disp,
                         p['x0'] / pix, p['y0'] / pix,
                         max(rf_map.shape) * 0.22)
    ax_f.text(0.04, 0.96, 'model', transform=ax_f.transAxes,
              fontsize=5.5, color='white', va='top',
              bbox=dict(facecolor='#1565C0', alpha=0.7, pad=1, edgecolor='none'))
    ax_f.set_title(
        f'σ={p["sigma"]:.1f}°  κ={p["kappa"]:.2f}  κ_dir={p["kappa_dir"]:.2f}\n'
        f'φ={phi_disp:.0f}°  R²={q["r_squared"]:.2f}',
        fontsize=6.2, color=r2_col, pad=2
    )

fig.suptitle(
    f'Mouse V1 m=1 edge-detector simple cells  (n={n_neurons}, all manually verified)\n'
    'Left: raw ridge RF map   |   Right: Gaussian derivative model (m=1, Lindeberg framework)\n'
    'σ = scale (°),  κ = elongation (max/min),  κ_dir = σ_orth/σ_φ (<1 = along differentiation),  φ = differentiation axis (lobe geometry),  R² = fit quality\n'
    'White bar = preferred edge orientation (φ+90°).  Sorted by κ ascending.',
    fontsize=10, y=1.02, color='#1a1a2e'
)

out = gallery_dir / 'fig_rf_gallery_m1_by_kappa.png'
plt.savefig(out, dpi=180, bbox_inches='tight', facecolor='white')
plt.savefig(out.with_name('fig_rf_gallery_m1_by_kappa_hires.png'),
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
phi_series = {cid: (float(phi_fresh)
                    if (phi_fresh is not None and not np.isnan(float(phi_fresh)))
                    else float(m1.loc[m1['cell_id']==cid, 'theta_hybrid'].iloc[0]))
              for cid, (p, q, fitted, pix, phi_fresh) in fit_results.items()}

plot_df2 = (
    m1[m1['cell_id'].isin(fit_results)].copy()
    .assign(phi_sort=lambda df: df['cell_id'].map(phi_series))
    .sort_values('phi_sort', ascending=True)
    .reset_index(drop=True)
)

fig2 = plt.figure(figsize=(N_COLS * 4.2, n_rows * 2.1), facecolor='white')
outer2 = gridspec.GridSpec(n_rows, N_COLS, figure=fig2, hspace=0.28, wspace=0.10)

for idx, row in plot_df2.iterrows():
    cid = int(row['cell_id'])
    p, q, fitted, pix, phi_fresh, phi_conf, *_ = fit_results[cid]
    rf_map = rf_map_cache[cid][0]
    name   = name_map.get(cid, '')

    inner2 = gridspec.GridSpecFromSubplotSpec(
        1, 2, subplot_spec=outer2[idx // N_COLS, idx % N_COLS], wspace=0.04)

    vmax = np.nanpercentile(np.abs(rf_map), 98)
    imkw = dict(cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                aspect='equal', origin='lower', interpolation='nearest')

    phi_disp = (float(phi_fresh)
                if (phi_fresh is not None and not np.isnan(float(phi_fresh)))
                else (float(row['theta_hybrid']) if pd.notna(row['theta_hybrid'])
                      else float(p['theta'])))
    r2_col = '#1a7a4a' if q['r_squared'] >= 0.5 else '#888888'

    ax_r = fig2.add_subplot(inner2[0, 0])
    ax_r.imshow(rf_map, **imkw)
    ax_r.set_xticks([]); ax_r.set_yticks([])
    for sp in ax_r.spines.values():
        sp.set_edgecolor('#bbbbbb'); sp.set_linewidth(0.5)
    ax_r.set_title(f'#{idx+1} {name}', fontsize=6.5, color='#444', pad=2, loc='left')
    ax_r.text(0.04, 0.96, 'raw RF', transform=ax_r.transAxes,
              fontsize=5.5, color='white', va='top',
              bbox=dict(facecolor='#333', alpha=0.65, pad=1, edgecolor='none'))

    ax_f = fig2.add_subplot(inner2[0, 1])
    ax_f.imshow(fitted, **imkw)
    ax_f.set_xticks([]); ax_f.set_yticks([])
    for sp in ax_f.spines.values():
        sp.set_edgecolor('#2196F3'); sp.set_linewidth(1.0)
    draw_orientation_bar(ax_f, phi_disp,
                         p['x0'] / pix, p['y0'] / pix,
                         max(rf_map.shape) * 0.22)
    ax_f.text(0.04, 0.96, 'model', transform=ax_f.transAxes,
              fontsize=5.5, color='white', va='top',
              bbox=dict(facecolor='#1565C0', alpha=0.7, pad=1, edgecolor='none'))
    ax_f.set_title(
        f'σ={p["sigma"]:.1f}°  κ={p["kappa"]:.2f}  κ_dir={p["kappa_dir"]:.2f}\n'
        f'φ={phi_disp:.0f}°  R²={q["r_squared"]:.2f}',
        fontsize=6.2, color=r2_col, pad=2
    )

fig2.suptitle(
    f'Mouse V1 m=1 edge-detector simple cells  (n={n_neurons}, all manually verified)\n'
    'Left: raw ridge RF map   |   Right: Gaussian derivative model (m=1, Lindeberg framework)\n'
    'σ = scale (°),  κ = elongation (max/min),  κ_dir = σ_orth/σ_φ (<1 = along differentiation),  φ = differentiation axis (lobe geometry),  R² = fit quality\n'
    'White bar = preferred edge orientation (φ+90°).  Sorted by φ ascending (0°→180°).',
    fontsize=9, y=1.02, color='#1a1a2e'
)

out2 = gallery_dir / 'fig_rf_gallery_m1_by_orientation.png'
plt.savefig(out2, dpi=180, bbox_inches='tight', facecolor='white')
plt.savefig(out2.with_name('fig_rf_gallery_m1_by_orientation_hires.png'),
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()